# **Instalando Bibliotecas**

In [27]:
# pip install thefuzz python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.8 MB/s eta 0:00:00


# **Importando bibliotecas**

In [1]:
import pandas as pd
import numpy as np
import joblib
import urllib.request
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

def drop_reset_index(df):
    df = df.dropna()
    df = df.reset_index(drop=True)
    df.index += 1
    return df

def safe_prob(column):
    return (1 / pd.to_numeric(column, errors='coerce').replace(0, np.nan)).fillna(0)

# **Data do jogo**

In [55]:
intervalo_dias = False # Mude para True se quiser escanear um período passado
dia_unico   = "2026-03-08"

data_inicio = "2026-03-01"
data_fim    = "2026-03-31"

# **DICIONÁRIO DE TRADUÇÃO DE LIGAS (Betfair -> Seu Modelo)**

In [56]:
tradutor_ligas = {
    "English Championship": "ENGLAND 2",
    "Belgian First Division A": "BELGIUM 1",
    "French Ligue 1": "FRANCE 1",
    "Italian Serie B": "ITALY 2",
    "Spanish Segunda Division": "SPAIN 2",
    "Dutch Eredivisie": "NETHERLANDS 1",
    "Swiss Super League": "SWITZERLAND 1",
    "Chilean Primera Division": "CHILE 1",
    "Chinese Super League": "CHINA 1",
    "South Korean K League 2": "SOUTH KOREA 2",
    "Scottish Championship": "SCOTLAND 2",
    "Danish Superliga": "DENMARK 1",
    "English League 2": "ENGLAND 4",
    "Slovakian Super League": "SLOVAKIA 1",
    "Irish Premier Division": "IRELAND 1"
}

# **CONFIGURAÇÕES DE FILTRO E CARREGAMENTO DO MODELO**

In [57]:
ODD_MAX_LAY = 4.00
FILTRO_EDGE = -0.70

# --- 3. CARREGAMENTO DO MODELO ---
url_modelo = 'https://github.com/tuedoidoe/Previsao_Entrada/raw/refs/heads/main/Dados_Excel/Modelo_LayAway_5.pkl'
caminho_local = 'Modelo_LayAway_5.pkl'

try:
    urllib.request.urlretrieve(url_modelo, caminho_local)
    dados_modelo = joblib.load(caminho_local)
    model = dados_modelo['modelo']
    taxas_ligas = dados_modelo['liga_rates']
    media_global_treino = dados_modelo['media_global']
    X_cols_treino = dados_modelo['features']
    ligas_autorizadas = dados_modelo.get('ligas_autorizadas', [])
    print("✅ Inteligência Artificial carregada com sucesso!")
except Exception as e:
    print(f"❌ Falha ao carregar modelo: {e}")

✅ Inteligência Artificial carregada com sucesso!


# **Requisição ao GitHub e criação**


In [58]:
# 1. Carregamento da Base Mãe para cálculos de Power_Diff
print("🧠 Carregando Base Histórica para cálculo de performance... (Aguarde)")
url_base_mae = "https://github.com/futpythontrader/Bases_de_Dados/raw/refs/heads/main/Base_de_Dados_BetfairExchange.csv"
df_hist = pd.read_csv(url_base_mae)
df_hist['Date'] = pd.to_datetime(df_hist['Date'])

# Função interna para calcular Power_Diff no momento exato do jogo
def calcular_performance_hist(row, df_hist):
    data_jogo = pd.to_datetime(row['Date'])
    time_h = row['Home']
    time_a = row['Away']

    # Filtra jogos que aconteceram ANTES da data do jogo atual
    past = df_hist[df_hist['Date'] < data_jogo]

    # Pega os últimos 8 jogos de cada time (Casa ou Fora)
    h_last = past[(past['Home'] == time_h) | (past['Away'] == time_h)].tail(8)
    a_last = past[(past['Home'] == time_a) | (past['Away'] == time_a)].tail(8)

    if len(h_last) < 3 or len(a_last) < 3:
        return 0.0 # Sem dados suficientes, neutraliza

    # Cálculo de Saldo de Gols Médio (Power)
    def get_avg_goals_diff(df, team):
        diffs = []
        for _, r in df.iterrows():
            if r['Home'] == team:
                diffs.append(r['Goals_H_FT'] - r['Goals_A_FT'])
            else:
                diffs.append(r['Goals_A_FT'] - r['Goals_H_FT'])
        return sum(diffs) / len(diffs)

    pwr_h = get_avg_goals_diff(h_last, time_h)
    pwr_a = get_avg_goals_diff(a_last, time_a)

    return (pwr_h - pwr_a) # Power_Diff: Positivo favorece Home, Negativo favorece Away

# --- Início da Coleta dos Jogos do Dia ---
base_url = "https://github.com/futpythontrader/Jogos_do_Dia/raw/refs/heads/main/Betfair/Jogos_do_Dia_Betfair_Back_Lay_{}.csv"
lista_dias = pd.date_range(data_inicio, data_fim).strftime("%Y-%m-%d") if intervalo_dias else [dia_unico]

dfs = []
print(f"⏳ Baixando arquivos diários de {len(lista_dias)} dias...")

for dia in lista_dias:
    try:
        df_temp = pd.read_csv(base_url.format(dia))
        df_temp["Date"] = dia
        dfs.append(df_temp)
    except:
        continue

if not dfs:
    print(f"🚨 Nenhum arquivo encontrado no intervalo {data_inicio} a {data_fim}.")
    df_var = pd.DataFrame()
else:
    Jogos_do_Dia = pd.concat(dfs, ignore_index=True)
    print(f"🌍 Total de jogos brutos encontrados: {len(Jogos_do_Dia)}")

    # Tradução das Ligas
    Jogos_do_Dia['League'] = Jogos_do_Dia['League'].map(tradutor_ligas).fillna(Jogos_do_Dia['League'])

    # Filtro de Ligas Autorizadas
    if ligas_autorizadas:
        df_filt = Jogos_do_Dia[Jogos_do_Dia['League'].str.upper().isin([l.upper() for l in ligas_autorizadas])].copy()
    else:
        df_filt = Jogos_do_Dia.copy()

    # Filtros Operacionais
    for col in ['Odd_A_Back', 'Odd_A_Lay', 'Odd_H_Back', 'Odd_H_Lay']:
        df_filt[col] = pd.to_numeric(df_filt[col], errors='coerce')

    df_filt = df_filt[
        (abs(df_filt['Odd_A_Back'] - df_filt['Odd_A_Lay']) <= 1.00) &
        (abs(df_filt['Odd_H_Back'] - df_filt['Odd_H_Lay']) <= 1.00) &
        (df_filt['Odd_A_Lay'] <= ODD_MAX_LAY) &
        (df_filt['Odd_H_Back'] < df_filt['Odd_A_Back'])
    ].dropna(subset=['Odd_A_Lay'])

    if df_filt.empty:
        print("⚠️ Nenhum jogo passou pelos filtros iniciais.")
        df_var = pd.DataFrame()
    else:
        df_var = drop_reset_index(df_filt)

        # --- Cálculo do Power_Diff Real (O "Pulo do Gato") ---
        print("🔄 Injetando inteligência histórica nos jogos selecionados...")
        df_var['Power_Diff'] = df_var.apply(lambda r: calcular_performance_hist(r, df_hist), axis=1)

        # Engenharia de Atributos do Dia
        for col in df_var.columns:
            if 'Odd' in col or 'CS' in col:
                df_var[col] = pd.to_numeric(df_var[col], errors='coerce')

        df_var['Prob_1x2_A'] = safe_prob(df_var['Odd_A_Back'])
        df_var['Prob_CS_Resistance'] = safe_prob(df_var['Odd_CS_1x0_Lay']) + safe_prob(df_var['Odd_CS_2x1_Lay'])
        df_var['Market_Asymmetry'] = (df_var['Prob_CS_Resistance'] - df_var['Prob_1x2_A'])
        df_var['Draw_Density'] = safe_prob(df_var['Odd_CS_0x0_Lay']) + safe_prob(df_var['Odd_CS_1x1_Lay'])
        df_var['Volatility_Risk'] = np.clip((safe_prob(df_var['Odd_Over25_FT_Back']) * df_var['Odd_A_Back']), 0, 50)

        # LIGA_RATE vindo do Treino
        df_var['LIGA_RATE'] = df_var['League'].map(taxas_ligas).fillna(media_global_treino)

        # Preenchimento de colunas ausentes (Garante compatibilidade com IA)
        for col in X_cols_treino:
            if col not in df_var.columns:
                df_var[col] = 3.0 if 'Odd' in col else 0.0

🧠 Carregando Base Histórica para cálculo de performance... (Aguarde)
⏳ Baixando arquivos diários de 1 dias...
🌍 Total de jogos brutos encontrados: 265
🔄 Injetando inteligência histórica nos jogos selecionados...


# **Jogos do Dia - API BetFair / GitHub**

In [66]:
print("\n" + "="*80)
periodo_msg = f"{data_inicio} até {data_fim}" if intervalo_dias else dia_unico
print(f"📊 PROCURANDO OPORTUNIDADES NO PERÍODO: {periodo_msg}")
print("="*80)

if df_var.empty:
    print("❌ Scanner encerrado: Não há jogos que atendam aos critérios no período selecionado.")
    if 'Jogos_do_Dia' in locals():
        # Caso não encontre nada, mostra as ligas disponíveis para conferência de nomes
        ligas_disp = Jogos_do_Dia['League'].unique()[:5]
        print(f"Dica: As ligas encontradas nos arquivos foram: {ligas_disp}")
else:
    # 1. Seleção das colunas exatas que o modelo espera (X_cols_treino)
    X_today = df_var[X_cols_treino]

    # 2. Geração das Probabilidades da IA (Chance de Sucesso do Lay Away)
    # Pegamos a segunda coluna [:, 1] que representa a classe 1 (Lucro/Green)
    df_var['Prob_IA'] = model.predict_proba(X_today)[:, 1]

    # 3. Cálculo da Probabilidade do Mercado (Baseada na Odd da Betfair)
    # Prob_Mercado = 1 - (1 / Odd_Lay). Isso representa a chance implícita do mercado.
    df_var['Prob_Mercado'] = 1 - (1 / df_var['Odd_A_Lay'].replace(0, np.nan))

    # 4. Cálculo do Edge (Vantagem Matemática)
    df_var['Edge'] = df_var['Prob_IA'] - df_var['Prob_Mercado']

    # # --- DIAGNÓSTICO DO MOTOR DE PREDIÇÃO ---
    # print("--- DIAGNÓSTICO DO EDGE (CAMINHO 2 - HISTÓRICO) ---")
    # print(f"Média Probabilidade IA: {df_var['Prob_IA'].mean():.2%}")
    # print(f"Média Probabilidade Mercado: {df_var['Prob_Mercado'].mean():.2%}")
    # print(f"Média Power_Diff Calculado: {df_var['Power_Diff'].mean():.4f}")
    # print(f"Média do Edge Real: {df_var['Edge'].mean():.2%}")
    # print("---------------------------------------------------\n")

    # 5. Filtro de Valor Final
    # Só entram jogos onde a nossa IA vê uma chance muito maior que a do mercado
    df_entradas = df_var[(df_var['Edge'] >= FILTRO_EDGE)].copy()

    if df_entradas.empty:
        print(f"💡 Foram analisados {len(df_var)} jogos.")
        print(f"Mesmo com dados históricos, nenhum jogo atingiu o Edge de {FILTRO_EDGE*100}%.")
        print("Isso pode ocorrer se o mercado estiver 'justo' ou se os times favoritos estiverem muito fortes.")
    else:
        # Seleção das colunas para exibição final
        colunas_final = ["Date", "Time", "League", "Home", "Away", "Odd_A_Lay"]
        df_entradas = df_entradas[colunas_final]

        # Ordenação
        df_entradas = df_entradas.sort_values(by=['Date', 'Time'], ascending=[True, True])

        # # Formatação estética
        # df_entradas['Edge'] = (df_entradas['Edge'] * 100).round(2).astype(str) + '%'

        # print(f"✅ SUCESSO! Encontradas {len(df_entradas)} entradas lucrativas no período.")
        display(drop_reset_index(df_entradas))


📊 PROCURANDO OPORTUNIDADES NO PERÍODO: 2026-03-08


,Date,Time,League,Home,Away,Odd_A_Lay
1,2026-03-08,10:00:00,SWITZERLAND 1,St Gallen,FC Basel,3.50
2,2026-03-08,10:30:00,NETHERLANDS 1,Fortuna Sittard,SC Telstar,3.40
3,2026-03-08,11:00:00,ITALY 2,Catanzaro,Empoli,3.55
4,2026-03-08,11:00:00,ITALY 2,Mantova,Juve Stabia,3.10
5,2026-03-08,12:15:00,SPAIN 2,Andorra CF,Sporting Gijon,3.25
6,2026-03-08,12:30:00,SWITZERLAND 1,Young Boys,Thun,2.90
7,2026-03-08,14:00:00,SLOVAKIA 1,Dunajska Streda,Podbrezova,4.00
8,2026-03-08,14:30:00,SPAIN 2,Racing Santander,Cordoba,3.80
9,2026-03-08,14:30:00,SPAIN 2,Leganes,Eibar,3.60
10,2026-03-08,15:30:00,ITALY 2,Pescara,SSD Bari,4.00


# **VALIDAÇÃO DE RESULTADOS (BACKTEST REAL)**

In [54]:
print("🏁 Iniciando Validação de Greens e Reds...")

# --- CORREÇÃO DE TIPO DE DADOS ---
# Garantindo que a coluna Date seja datetime em ambos os dataframes para o merge funcionar
df_entradas['Date'] = pd.to_datetime(df_entradas['Date'])
df_hist['Date'] = pd.to_datetime(df_hist['Date'])

from thefuzz import process

# 1. Obter listas únicas de times de ambos os lados
times_entradas = df_entradas['Home'].unique().tolist() + df_entradas['Away'].unique().tolist()
times_hist = df_hist['Home'].unique().tolist() + df_hist['Away'].unique().tolist()

# 2. Função que encontra o nome mais parecido no histórico
def encontrar_match(nome, lista_historico, threshold=50):
    match, score = process.extractOne(nome, lista_historico)
    return match if score >= threshold else nome

# 3. Criar um dicionário de mapeamento (Isso acelera o processo)
mapa_times = {time: encontrar_match(time, times_hist) for time in times_entradas}

# 4. Normalizar os DataFrames usando o mapa
df_entradas['Home_Match'] = df_entradas['Home'].map(mapa_times)
df_entradas['Away_Match'] = df_entradas['Away'].map(mapa_times)

# 5. Fazer o Merge com as colunas corrigidas
df_backtest = pd.merge(
    df_entradas,
    df_hist[['Date', 'Home', 'Away', 'Goals_H_FT', 'Goals_A_FT']],
    left_on=['Date', 'Home_Match', 'Away_Match'],
    right_on=['Date', 'Home', 'Away'],
    how='left',
    suffixes=('', '_hist')
)

# # Filtra apenas o que não deu match
# nao_deu_match = df_backtest[df_backtest['Goals_H_FT'].isna()]
# print("Lista de times que não deram match:")
# print(nao_deu_match[['Date', 'Home', 'Away']].head(50))

# 2. Definição de Green ou Red (Regra: Lay Away)
def verificar_resultado(row):
    if pd.isna(row['Goals_H_FT']): return "Sem Dados"

    # No Lay Away: Se o Away (Visitante) NÃO ganhar, é GREEN.
    # Se o Away ganhar (Gols Away > Gols Home), é RED.
    if row['Goals_H_FT'] >= row['Goals_A_FT']:
        return "GREEN"
    else:
        return "RED"

df_backtest['Resultado'] = df_backtest.apply(verificar_resultado, axis=1)

# 3. Cálculo Financeiro
# No Lay Away a Responsabilidade varia, mas o lucro é fixo.
stake = 50
def calcular_pl(row):
    if row['Resultado'] == "RED":
        return -stake
    elif row['Resultado'] == "GREEN":
        odd = float(row['Odd_A_Lay'])
        return stake / (odd - 1) * 0.95
    return 0

df_backtest['P/L'] = df_backtest.apply(calcular_pl, axis=1)

# --- RELATÓRIO FINAL ---
df_validos = df_backtest[df_backtest['Resultado'] != "Sem Dados"]
total_jogos = len(df_validos)
greens = len(df_validos[df_validos['Resultado'] == "GREEN"])
reds = len(df_validos[df_validos['Resultado'] == "RED"])
win_rate = (greens / total_jogos) * 100 if total_jogos > 0 else 0
lucro_total = df_validos['P/L'].sum()

print("\n" + "="*50)
print(f"🏆 RELATÓRIO DE PERFORMANCE")
print("="*50)
print(f"✅ Total de Entradas: {total_jogos}")
print(f"🟢 Greens: {greens}")
print(f"🔴 Reds: {reds}")
print(f"📈 Taxa de Acerto: {win_rate:.2f}%")
print(f"💰 Lucro/Prejuízo Total: R$ {lucro_total:.2f}")
print("="*50)

# Mostrar os 50 primeiros resultados para conferência
display(df_backtest[['Date', 'Home', 'Away', 'Odd_A_Lay', 'Goals_H_FT', 'Goals_A_FT', 'Resultado', 'P/L']].head(50))

🏁 Iniciando Validação de Greens e Reds...

🏆 RELATÓRIO DE PERFORMANCE
✅ Total de Entradas: 13
🟢 Greens: 11
🔴 Reds: 2
📈 Taxa de Acerto: 84.62%
💰 Lucro/Prejuízo Total: R$ 133.91


,Date,Home,Away,Odd_A_Lay,Goals_H_FT,Goals_A_FT,Resultado,P/L
0,2026-03-01,Fredericia,Silkeborg,2.88,2.0,1.0,GREEN,25.265957
1,2026-03-01,Viborg,FC Nordsjaelland,2.98,2.0,1.0,GREEN,23.989899
2,2026-03-01,FC Twente,Feyenoord,3.05,2.0,0.0,GREEN,23.170732
3,2026-03-01,SonderjyskE,OB,3.30,NaN,NaN,Sem Dados,0.000000
4,2026-03-01,Excelsior,Go Ahead Eagles,3.00,NaN,NaN,Sem Dados,0.000000
5,2026-03-01,Mantova,Carrarese,3.30,1.0,1.0,GREEN,20.652174
6,2026-03-01,Colo Colo,Universidad de Chile,3.35,0.0,1.0,RED,-50.000000
7,2026-03-01,Paris FC,Nice,3.30,1.0,0.0,GREEN,20.652174
8,2026-03-01,FC Utrecht,Az Alkmaar,3.50,2.0,0.0,GREEN,19.000000
9,2026-03-03,Padova,Spezia,3.05,2.0,2.0,GREEN,23.170732


In [24]:
# df_hist.to_excel(f'hist.xlsx', index=False)